In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import pygris

import geopandas as gpd
import pickle 

pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None


In [ ]:
tract_cols = [
    'NAME',
    'ALAND',
    'AWATER',
    'geometry',
]

# Downloading 2010 and 2020 tracts for Cuyahoga County, Ohio.
tracts_2010 = pygris.tracts(state="39", county="035", year=2015)[tract_cols]
tracts_2010['NAME'] = tracts_2010['NAME'].astype(str)

tracts_2020 = pygris.tracts(state="39", county="035", year=2020)[tract_cols]
tracts_2020['NAME'] = tracts_2020['NAME'].astype(str)

# Excluding tract 9900, which represents a water geometry over lake erie
tracts_2010 = tracts_2010[tracts_2010['NAME'] != '9900']
tracts_2020 = tracts_2020[tracts_2020['NAME'] != '9900']

In [ ]:
def convert_tract_to_fips(tract_id): 
    tract_str = str(tract_id).strip()

    if "." in tract_str:
        before_decimal, after_decimal = tract_str.split(".")
        before_decimal = before_decimal.zfill(4)
        after_decimal = after_decimal.ljust(2, "0")
    else:
        before_decimal = tract_str.zfill(4)
        after_decimal = "00"

    return f"39035{before_decimal}{after_decimal}"

In [ ]:
tracts_2010["NAME"] = tracts_2010['NAME'].apply(convert_tract_to_fips)

In [ ]:
tracts_2020["NAME"] = tracts_2020['NAME'].apply(convert_tract_to_fips)

In [ ]:
count_overlap = 0
for i in tracts_2010["NAME"].unique():
    if i in tracts_2020["NAME"].unique():
        count_overlap += 1
count_overlap

There are 381 tract names in 2010 that match with 2020. However, there are over 60 tracts that are not accounted for within this. Additionally within the 381 tracts that do match in names, there are some that have grown in area.

In [ ]:
tracts_allign_df = pd.DataFrame(columns=['NAME_10','NAME_20','OL_PROP'])

# Function handling the calculation of the overlap area
ol_area_fn = lambda intersect_row: (row.geometry.intersection(intersect_row.geometry).area / row.geometry.area)

for i in range(tracts_2020.shape[0]):
    row = tracts_2020.iloc[i]
    intersect = tracts_2010[tracts_2010.geometry.intersects(row.geometry)]

    intersect['NAME_20'] = row['NAME']
    intersect['OL_PROP'] = intersect.apply(ol_area_fn, axis=1)
    intersect = intersect.drop(columns=['geometry','ALAND','AWATER']).rename(columns={'NAME': 'NAME_10'})
    intersect = intersect[intersect['OL_PROP'] > 0.08]

    tracts_allign_df = pd.concat([tracts_allign_df, intersect])

# Adding a found special case to the dataframe
tracts_allign_df.loc[len(tracts_allign_df)] = ['1948', '1971', 1.0]

In [ ]:
tracts_allign_df

In [ ]:
tracts_allign_df['NAME_10'].value_counts().head(50)

In [ ]:
# Get number of unique 2010 and 2020 tracts
num_2010 = len(tracts_allign_df['NAME_10'].unique())
num_2020 = len(tracts_allign_df['NAME_20'].unique())

print(num_2010,num_2020)
print(tracts_2010['NAME'].unique().shape[0],tracts_2020['NAME'].unique().shape[0])

In [ ]:
tracts_2010_that_dont_merge = tracts_allign_df['NAME_10'].value_counts()
lst_tracts_2010_good = list(tracts_2010_that_dont_merge[tracts_2010_that_dont_merge == 1].index)

In [ ]:
tracts_2020_that_dont_merge = tracts_allign_df['NAME_20'].value_counts()
lst_tracts_2020_good = list(tracts_2020_that_dont_merge[tracts_2020_that_dont_merge == 1].index)

In [ ]:
tracts_2010.head()

In [ ]:
cleaned_tracts_2010 = tracts_2010[tracts_2010['NAME'].isin(lst_tracts_2010_good)]
cleaned_tracts_2020 = tracts_2020[tracts_2020['NAME'].isin(lst_tracts_2020_good)]

In [ ]:
tracts_allign_df['NAME_10'].value_counts().head(20)

In [ ]:
changes_df = pd.read_csv("https://www2.census.gov/geo/docs/maps-data/data/rel2020/tract/tab20_tract20_tract10_natl.txt", delimiter="|", dtype=str)
changes_df = changes_df[changes_df['GEOID_TRACT_20'].str.startswith("39035")]
changes_df.head()

In [ ]:
# We will create a function that creates a dicts that consists of tract ids from 2020 as keys and 2010 tract ids as values. 
# It will only consists of tracts that do NOT consist from merging or splitting. It has to be the exact same tract. 
# The AREALAND_TRACT_20 and AREALAND_TRACT_10 are the exact same 
good_tract_ids = {}
for i in range(changes_df.shape[0]):
    row = changes_df.iloc[i]
    if row["AREALAND_TRACT_20"] == row["AREALAND_TRACT_10"]:
        if row["GEOID_TRACT_20"] != row["GEOID_TRACT_10"]:
            print(f"THIS ID IS WEIRD:")
        good_tract_ids[row["GEOID_TRACT_20"]] = row["GEOID_TRACT_10"]

In [ ]:
good_tract_ids

In [ ]:
# save as a pickle file
with open('data/pickle_files/good_tract_ids.pickle', 'wb') as handle:
    pickle.dump(good_tract_ids, handle, protocol=pickle.HIGHEST_PROTOCOL)
    

In [ ]:
# load pickle file
with open('data/pickle_files/good_tract_ids.pickle', 'rb') as handle:
    good_tract_ids = pickle.load(handle)

In [ ]:
t_dict = {}
for index, row in same_tracts_df.iterrows():
    t_dict[row['NAME_10']] = row['NAME_20']
for index, row in diff_tracts_df.iterrows():
    t_dict[row['NAME_10']] = row['NAME_20']

In [ ]:
tracts_2010_new = tracts_2010.copy()
tracts_2010_new['NAME'] = tracts_2010_new['NAME'].map(t_dict)